In [ ]:
import os
from pyspark.sql import functions as F
from src.utils.config import *
from src.utils.kafka import create_kafka_consumer, insert_to_clickhouse
from src.utils.spark import get_spark

spark = get_spark()

bootstrap_servers = os.getenv('KAFKA_BOOTSTRAP_SERVERS')
ch_user = os.getenv('CLICKHOUSE_USER')
ch_password = os.getenv('CLICKHOUSE_PW')
ch_host = os.getenv('CLICKHOUSE_HOST')
client = setup_clickhouse_client(ch_user, ch_password, ch_host)

consumer_instance = create_kafka_consumer(
    client_id='weather_processor',
    bootstrap_servers=bootstrap_servers,
    topic_name='raw_weather_data',
)

In [ ]:
col_names=["date_observed", "current_temp", "feels_like_temp", "pressure_hPa", "humidity_pct", "min_temp", "max_temp", "wind_speed_ms", 
           "wind_deg", "cloudiness_pct", "rain_1h", "rain_3h", "weather_main", "weather_desc"]

defaults = {
    "date_observed": "2099-12-31t00:00:00.000",
    "current_temp": 999.99,
    "feels_like_temp": 999.99,
    "pressure_hPa": -100,
    "humidity_pct": -100,
    "min_temp": -999.99,
    "max_temp": 999.99,
    "wind_speed_ms": 0.00,
    "wind_deg": 0,
    "cloudiness_pct": -100,
    "rain_1h": 0.00,
    "rain_3h": 0.00,
    "weather_main": [],
    "weather_desc": [],
}

insert_to_clickhouse(consumer_instance, 'historical_weather_data', client)